# 데이터 전처리 (Data Preprocessing)

## customers.csv 수치화 변환

**목적**: 머신러닝 모델은 수치 데이터만 처리할 수 있습니다. 따라서 범주형(문자열) 데이터를 적절한 인코딩 기법으로 수치화해야 합니다.

이 노트북에서는 각 변환의 **과학적 원리**를 상세히 설명하며 전처리를 진행합니다.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

print("라이브러리 로드 완료")

라이브러리 로드 완료


---
## 1. 데이터 로드 및 탐색적 데이터 분석 (EDA)

데이터 전처리의 첫 번째 단계는 데이터의 구조와 특성을 파악하는 것입니다.
- **형태(shape)**: 행(샘플 수)과 열(특성 수) 확인
- **타입(dtypes)**: 각 컬럼의 데이터 타입 확인 → `object` 타입이 범주형 후보
- **기술통계(describe)**: 수치형 컬럼의 분포 파악
- **고유값(unique)**: 범주형 컬럼의 카테고리 확인

In [2]:
df = pd.read_csv("customers.csv")
print(f"데이터 형태: {df.shape[0]}행 x {df.shape[1]}열")
df.head()

데이터 형태: 10000행 x 11열


,고객ID,성별,결제수단,거주지,회원등급,만족도,최근접속시간(시),선호제품군_적정온도,나이,구매수량,총결제금액
0,CUST_0001,남성,휴대폰결제,대구,Gold,2,1,27.0,34,7,760355
1,CUST_0002,여성,계좌이체,대구,Gold,5,15,24.6,20,42,727001
2,CUST_0003,여성,신용카드,광주,Gold,1,6,20.3,51,8,618787
3,CUST_0004,여성,계좌이체,서울,VVIP,5,20,16.8,19,29,434545
4,CUST_0005,여성,계좌이체,광주,Gold,3,6,23.5,53,45,604750


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   고객ID        10000 non-null  object 
 1   성별          10000 non-null  object 
 2   결제수단        10000 non-null  object 
 3   거주지         10000 non-null  object 
 4   회원등급        10000 non-null  object 
 5   만족도         10000 non-null  int64  
 6   최근접속시간(시)   10000 non-null  int64  
 7   선호제품군_적정온도  10000 non-null  float64
 8   나이          10000 non-null  int64  
 9   구매수량        10000 non-null  int64  
 10  총결제금액       10000 non-null  int64  
dtypes: float64(1), int64(5), object(5)
memory usage: 859.5+ KB


In [4]:
df.describe()

,만족도,최근접속시간(시),선호제품군_적정온도,나이,구매수량,총결제금액
count,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,2.998000,11.565500,19.95989,44.479000,25.701200,507879.636800
std,1.416685,6.926363,5.01381,15.014696,14.483484,284442.229407
min,1.000000,0.000000,0.40000,19.000000,1.000000,10026.000000
25%,2.000000,6.000000,16.60000,32.000000,13.000000,263423.500000
50%,3.000000,11.000000,19.90000,44.000000,26.000000,510927.500000
75%,4.000000,18.000000,23.30000,57.000000,38.000000,753943.000000
max,5.000000,23.000000,39.60000,70.000000,50.000000,999990.000000


In [5]:
# 범주형 컬럼 고유값 확인
for col in df.select_dtypes(include="object").columns:
    print(f"\n[{col}] 고유값 ({df[col].nunique()}개): {df[col].unique()}")


[고객ID] 고유값 (10000개): ['CUST_0001' 'CUST_0002' 'CUST_0003' ... 'CUST_9998' 'CUST_9999'
 'CUST_10000']

[성별] 고유값 (3개): ['남성' '여성' '기타']

[결제수단] 고유값 (4개): ['휴대폰결제' '계좌이체' '신용카드' '간편결제']

[거주지] 고유값 (5개): ['대구' '광주' '서울' '경기' '부산']

[회원등급] 고유값 (5개): ['Gold' 'VVIP' 'VIP' 'Bronze' 'Silver']


### 컬럼 분류

| 구분 | 컬럼 | 이유 |
|------|------|------|
| **식별자** | 고객ID | 고유 식별값 → 제거 대상 |
| **범주형 (명목)** | 성별, 결제수단, 거주지 | 순서 없는 카테고리 |
| **범주형 (순서)** | 회원등급 | Bronze < Silver < Gold < VIP < VVIP 순서 존재 |
| **수치형** | 만족도, 최근접속시간(시), 선호제품군_적정온도, 나이, 구매수량, 총결제금액 | 이미 숫자 |

> **왜 이 구분이 중요한가?**  
> 각 타입마다 적합한 인코딩 방법이 다릅니다. 잘못된 인코딩은 모델에 **거짓 정보**를 주입하여 성능을 저하시킵니다.

---
## 2. 고객ID 제거

### 과학적 근거: 정보 이론 (Information Theory)

고객ID는 각 행을 고유하게 식별하는 값입니다. 정보 이론 관점에서 분석하면:

- **엔트로피(Entropy)**: $H(X) = -\sum p(x) \log p(x)$
- 고유값이 N개이고 모두 1번씩 등장하면, 각 확률 $p = 1/N$
- 이 경우 엔트로피가 **최대**가 되어, 변수가 가진 "정보"가 최대입니다

그러나 이 "정보"는 **개별 샘플을 구분하는 정보**이지, **패턴을 학습하는 데 유용한 정보가 아닙니다.**

머신러닝 모델이 고객ID를 학습하면:
1. 훈련 데이터의 ID를 **암기(memorization)** 합니다
2. 새로운 데이터(테스트)에는 해당 ID가 없으므로 **일반화 실패**
3. 이것이 바로 **과적합(Overfitting)** 의 전형적 원인입니다

> **결론**: 고유 식별자는 예측에 기여하지 못하므로 반드시 제거합니다.

In [6]:
print(f"고객ID 고유값 수: {df['고객ID'].nunique()} / 전체 행 수: {len(df)}")
print(f"고유 비율: {df['고객ID'].nunique() / len(df) * 100:.1f}%")
print("\n→ 고유 비율 100%: 모든 값이 유일 → 패턴 학습 불가능 → 제거")

df = df.drop(columns=["고객ID"])
print(f"\n제거 후 컬럼: {list(df.columns)}")

고객ID 고유값 수: 10000 / 전체 행 수: 10000
고유 비율: 100.0%

→ 고유 비율 100%: 모든 값이 유일 → 패턴 학습 불가능 → 제거

제거 후 컬럼: ['성별', '결제수단', '거주지', '회원등급', '만족도', '최근접속시간(시)', '선호제품군_적정온도', '나이', '구매수량', '총결제금액']


---
## 3. 성별 — Label Encoding

### Label Encoding이란?

각 카테고리에 정수를 할당하는 가장 단순한 인코딩입니다.

$$\text{남성} \rightarrow 0, \quad \text{여성} \rightarrow 1, \quad \text{기타} \rightarrow 2$$

### 왜 Label Encoding을 사용하는가?

1. **저차원 범주**: 카테고리 수가 적을 때(2~3개) 효과적
2. **트리 기반 모델**: 결정 트리, 랜덤 포레스트 등은 분할(split) 기반이므로 순서 관계를 부여해도 문제없음
3. **메모리 효율**: One-Hot은 컬럼 수를 늘리지만, Label Encoding은 1개 컬럼 유지

### 주의사항

Label Encoding은 숫자 간 **크기 관계**를 암묵적으로 부여합니다:
- $0 < 1 < 2$ → "남성 < 여성 < 기타"라는 관계가 생김
- **선형 모델**(Linear Regression, Logistic Regression)은 이 거짓 순서를 학습할 위험이 있음
- **트리 모델**은 각 값을 독립적으로 분할하므로 이 문제가 없음

> 본 데이터에서는 **범주 수가 적고(3개)**, 트리 모델에서의 활용을 가정하여 Label Encoding을 적용합니다.

In [7]:
print("변환 전:")
print(df["성별"].value_counts())
print(f"dtype: {df['성별'].dtype}")

le_gender = LabelEncoder()
df["성별"] = le_gender.fit_transform(df["성별"])

print("\n변환 후:")
print(df["성별"].value_counts())
print(f"dtype: {df['성별'].dtype}")
print(f"\n매핑: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}")

변환 전:
성별
여성    3400
기타    3320
남성    3280
Name: count, dtype: int64
dtype: object

변환 후:
성별
2    3400
0    3320
1    3280
Name: count, dtype: int64
dtype: int64

매핑: {'기타': np.int64(0), '남성': np.int64(1), '여성': np.int64(2)}


---
## 4. 결제수단 — One-Hot Encoding

### One-Hot Encoding이란?

각 카테고리를 **이진 벡터(binary vector)** 로 표현합니다. N개 카테고리가 있으면 N개(또는 N-1개)의 새로운 컬럼이 생성됩니다.

| 원본 | 신용카드 | 계좌이체 | 휴대폰결제 | 간편결제 |
|------|:--------:|:--------:|:----------:|:--------:|
| 신용카드 | 1 | 0 | 0 | 0 |
| 계좌이체 | 0 | 1 | 0 | 0 |
| 휴대폰결제 | 0 | 0 | 1 | 0 |
| 간편결제 | 0 | 0 | 0 | 1 |

### 수학적 원리: 직교 벡터 공간

각 카테고리는 N차원 공간에서 **직교(orthogonal)** 하는 단위 벡터로 매핑됩니다:
- 신용카드 = $[1, 0, 0, 0]$
- 계좌이체 = $[0, 1, 0, 0]$

두 벡터의 **유클리드 거리**:
$$d = \sqrt{(1-0)^2 + (0-1)^2 + 0 + 0} = \sqrt{2}$$

모든 카테고리 쌍의 거리가 $\sqrt{2}$로 **동일**합니다. 이는 "어떤 결제수단도 다른 것보다 가깝거나 멀지 않다"는 명목형 변수의 본질을 정확히 반영합니다.

### 왜 Label Encoding이 아닌가?

Label Encoding을 적용하면:
- 신용카드=0, 계좌이체=1, 휴대폰결제=2, 간편결제=3
- 모델이 $|0-3| > |0-1|$로 학습 → "신용카드와 간편결제는 멀고, 계좌이체와는 가깝다"
- 이것은 **거짓 관계**입니다. 결제수단 간에는 순서나 거리가 존재하지 않습니다.

### 더미 변수 함정 (Dummy Variable Trap)

N개 카테고리를 N개 컬럼으로 만들면 **다중공선성(Multicollinearity)** 이 발생합니다:
- 3개 컬럼의 합이 항상 1 → 하나의 컬럼은 나머지로 완벽히 예측 가능
- 선형 회귀에서 역행렬 계산 불가 → 해가 불안정

`pd.get_dummies(drop_first=True)`로 첫 번째 카테고리를 제거하면 해결됩니다.

> 본 노트북에서는 **전체 카테고리를 유지**합니다 (트리 모델에서는 다중공선성이 문제되지 않음).

In [8]:
print("변환 전:")
print(df["결제수단"].value_counts())

df = pd.get_dummies(df, columns=["결제수단"], prefix="결제수단")

print("\n변환 후 (새로 생성된 컬럼):")
payment_cols = [c for c in df.columns if c.startswith("결제수단_")]
print(df[payment_cols].head(10))
print(f"\n생성된 컬럼 수: {len(payment_cols)}개")

변환 전:
결제수단
계좌이체     2548
간편결제     2542
휴대폰결제    2515
신용카드     2395
Name: count, dtype: int64

변환 후 (새로 생성된 컬럼):
   결제수단_간편결제  결제수단_계좌이체  결제수단_신용카드  결제수단_휴대폰결제
0      False      False      False        True
1      False       True      False       False
2      False      False       True       False
3      False       True      False       False
4      False       True      False       False
5      False       True      False       False
6       True      False      False       False
7      False      False       True       False
8      False      False       True       False
9       True      False      False       False

생성된 컬럼 수: 4개


---
## 5. 거주지 — One-Hot Encoding

### 적용 이유

거주지(대구, 광주, 서울, 경기, 부산)도 결제수단과 동일한 **명목형 변수**입니다:
- 도시 간에 내재적 순서가 없음 ("서울 > 부산"이라는 관계 없음)
- 각 도시는 동등한 카테고리
- 직교 벡터 공간으로 매핑하여 모든 도시 쌍의 거리를 동일하게 유지

결제수단과 **동일한 과학적 논리**가 적용되므로, 같은 방법(One-Hot Encoding)을 사용합니다.

In [9]:
print("변환 전:")
print(df["거주지"].value_counts())

df = pd.get_dummies(df, columns=["거주지"], prefix="거주지")

print("\n변환 후 (새로 생성된 컬럼):")
region_cols = [c for c in df.columns if c.startswith("거주지_")]
print(df[region_cols].head(10))
print(f"\n생성된 컬럼 수: {len(region_cols)}개")

변환 전:
거주지
경기    2069
광주    2023
대구    2005
부산    1953
서울    1950
Name: count, dtype: int64

변환 후 (새로 생성된 컬럼):
   거주지_경기  거주지_광주  거주지_대구  거주지_부산  거주지_서울
0   False   False    True   False   False
1   False   False    True   False   False
2   False    True   False   False   False
3   False   False   False   False    True
4   False    True   False   False   False
5    True   False   False   False   False
6   False    True   False   False   False
7   False   False    True   False   False
8   False   False    True   False   False
9   False   False   False    True   False

생성된 컬럼 수: 5개


---
## 6. 회원등급 — Ordinal Encoding (순서 인코딩)

### Ordinal Encoding이란?

카테고리에 **순서를 반영한 정수**를 할당합니다.

$$\text{Bronze} \rightarrow 0 < \text{Silver} \rightarrow 1 < \text{Gold} \rightarrow 2 < \text{VIP} \rightarrow 3 < \text{VVIP} \rightarrow 4$$

### 왜 One-Hot이 아닌가?

회원등급은 **순서형(Ordinal)** 변수입니다:
- Bronze < Silver < Gold < VIP < VVIP 라는 **내재적 순서**가 존재
- 이 순서는 비즈니스 도메인에서 정의된 것 (구매량, 충성도 기반)

One-Hot Encoding을 적용하면:
- 각 등급이 독립 벡터 → 순서 정보 **완전 소실**
- "VVIP가 Bronze보다 높은 등급"이라는 정보를 모델이 학습 불가

Ordinal Encoding은 이 순서를 수치에 **보존**합니다.

### 주의: LabelEncoder vs 명시적 매핑

`LabelEncoder`는 **알파벳 순서**로 정렬합니다:
- Bronze=0, Gold=1, Silver=2, VIP=3, VVIP=4
- Gold(1) < Silver(2) → **실제 순서와 불일치!**

따라서 **도메인 지식에 기반한 명시적 매핑**을 사용해야 합니다.

In [10]:
print("변환 전:")
print(df["회원등급"].value_counts())

# 도메인 지식 기반 순서 매핑
grade_order = {
    "Bronze": 0,
    "Silver": 1,
    "Gold": 2,
    "VIP": 3,
    "VVIP": 4
}

df["회원등급"] = df["회원등급"].map(grade_order)

print("\n변환 후:")
print(df["회원등급"].value_counts().sort_index())
print(f"dtype: {df['회원등급'].dtype}")
print(f"\n매핑: {grade_order}")

변환 전:
회원등급
Silver    2043
Gold      2038
VIP       2010
VVIP      1961
Bronze    1948
Name: count, dtype: int64

변환 후:
회원등급
0    1948
1    2043
2    2038
3    2010
4    1961
Name: count, dtype: int64
dtype: int64

매핑: {'Bronze': 0, 'Silver': 1, 'Gold': 2, 'VIP': 3, 'VVIP': 4}


---
## 7. 수치형 컬럼 확인 및 결측치 처리

### 결측치(Missing Value)가 문제인 이유

대부분의 머신러닝 알고리즘은 결측치를 처리할 수 없습니다:
- **수학적 이유**: NaN과의 연산 결과는 항상 NaN → 손실 함수 계산 불가
- **통계적 이유**: 결측 패턴이 무작위가 아닐 수 있음 (MCAR/MAR/MNAR)

### 처리 전략 비교

| 전략 | 장점 | 단점 | 적합한 경우 |
|------|------|------|------------|
| **행 삭제** | 편향 없음 | 데이터 손실 | 결측 비율 < 5% |
| **평균 대체** | 전체 평균 유지 | 분산 축소, 관계 왜곡 | 정규 분포 |
| **중앙값 대체** | 이상치에 강건 | 관계 왜곡 가능 | 비대칭 분포 |
| **최빈값 대체** | 범주형에 적합 | 수치형에 부적합 | 범주형 변수 |

> 본 노트북에서는 **중앙값(median) 대체**를 사용합니다. 이상치에 강건하기 때문입니다.

In [11]:
# 결측치 확인
print("=== 결측치 현황 ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({"결측수": missing, "비율(%)": missing_pct})
print(missing_df[missing_df["결측수"] > 0])

if missing.sum() == 0:
    print("\n결측치 없음 — 추가 처리 불필요")
else:
    print(f"\n총 결측치: {missing.sum()}개")
    # 수치형 결측치는 중앙값으로 대체
    numeric_cols = df.select_dtypes(include=["number"]).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"  {col}: 중앙값({median_val})으로 대체 완료")
    print(f"\n처리 후 총 결측치: {df.isnull().sum().sum()}개")

=== 결측치 현황 ===
Empty DataFrame
Columns: [결측수, 비율(%)]
Index: []

결측치 없음 — 추가 처리 불필요


---
## 8. 최종 검증

전처리의 목표는 **모든 컬럼이 수치형(numeric)**이 되는 것입니다.

검증 기준:
1. `object` 타입 컬럼이 **0개**인가?
2. 결측치가 **0개**인가?
3. 모든 값이 **유한(finite)**한가?

In [12]:
print("=== 최종 데이터 타입 검증 ===")
print(df.dtypes)
print(f"\nobject 타입 컬럼 수: {(df.dtypes == 'object').sum()}개")
print(f"결측치 수: {df.isnull().sum().sum()}개")
print(f"최종 데이터 형태: {df.shape[0]}행 x {df.shape[1]}열")

=== 최종 데이터 타입 검증 ===
성별              int64
회원등급            int64
만족도             int64
최근접속시간(시)       int64
선호제품군_적정온도    float64
나이              int64
구매수량            int64
총결제금액           int64
결제수단_간편결제        bool
결제수단_계좌이체        bool
결제수단_신용카드        bool
결제수단_휴대폰결제       bool
거주지_경기           bool
거주지_광주           bool
거주지_대구           bool
거주지_부산           bool
거주지_서울           bool
dtype: object

object 타입 컬럼 수: 0개
결측치 수: 0개
최종 데이터 형태: 10000행 x 17열


In [13]:
df.head(10)

,성별,회원등급,만족도,최근접속시간(시),선호제품군_적정온도,나이,구매수량,총결제금액,결제수단_간편결제,결제수단_계좌이체,결제수단_신용카드,결제수단_휴대폰결제,거주지_경기,거주지_광주,거주지_대구,거주지_부산,거주지_서울
0,1,2,2,1,27.0,34,7,760355,False,False,False,True,False,False,True,False,False
1,2,2,5,15,24.6,20,42,727001,False,True,False,False,False,False,True,False,False
2,2,2,1,6,20.3,51,8,618787,False,False,True,False,False,True,False,False,False
3,2,4,5,20,16.8,19,29,434545,False,True,False,False,False,False,False,False,True
4,2,2,3,6,23.5,53,45,604750,False,True,False,False,False,True,False,False,False
5,0,2,4,22,22.0,22,18,554530,False,True,False,False,True,False,False,False,False
6,1,3,1,3,24.5,41,11,83149,True,False,False,False,False,True,False,False,False
7,2,3,3,10,23.2,70,37,523660,False,False,True,False,False,False,True,False,False
8,2,0,2,12,25.2,25,36,58966,False,False,True,False,False,False,True,False,False
9,2,4,2,15,17.3,27,49,792361,True,False,False,False,False,False,False,True,False


---
## 9. 전처리 요약

| 컬럼 | 원본 타입 | 인코딩 방법 | 결과 | 과학적 근거 |
|------|----------|------------|------|------------|
| 고객ID | 문자열 | **제거** | - | 고유 식별자 → 엔트로피 최대 → 과적합 유발 |
| 성별 | 문자열 | **Label Encoding** | 정수 | 저차원 범주(3개), 트리 모델에 적합 |
| 결제수단 | 문자열 | **One-Hot Encoding** | 이진 벡터 | 명목형 → 직교 벡터로 등거리 보장 |
| 거주지 | 문자열 | **One-Hot Encoding** | 이진 벡터 | 명목형 → 동일 논리 적용 |
| 회원등급 | 문자열 | **Ordinal Encoding** | 순서 정수 | 순서형 → 내재적 순서 보존 |
| 수치형 컬럼 | 숫자 | 결측치 처리 | 숫자 | 중앙값 대체 (이상치 강건) |

**핵심 원칙**: 데이터의 본질(명목/순서/수치)에 맞는 인코딩을 선택해야 모델이 올바른 관계를 학습합니다.